# EU short-stay accommodation (online platforms)

#### Data Sources (Eurostat):

- `data/raw/tour_ce_omn12__custom_23247212_spreadsheet.xlsx`: Short-stay accommodation offered via online platforms by months, residence of the guest and NUTS 1 and NUTS 2 region, nights spent, 2018–2026.
- `data/raw/NUTS2021-NUTS2024.xlsx`: NUTS codes, NUTS 3 mountain and island typologies.
- `data/raw/EU-27-LAU-2024-NUTS-2024.xlsx`: DEGURBA, coastal areas.

This notebook builds and tests the **clean dataset**, `data/transformed/tour_ce_omn12_long.csv`, from the raw Eurostat files: nights spent by region, year, month and residence of the guest, with NUTS codes, the region hierarchy and each region's geography. No indices are calculated here; they are added and explained in `Stage 2 Indices.ipynb`, which reads this file; the findings are in `Stage 3 Analysis.ipynb`.

| Column | Meaning |
|---|---|
| Region | Country name (level 0) or NUTS region name as published by Eurostat |
| NUTSCode | NUTS 2024 code (country code for level 0, `EL` = Greece) — the unique key of a region |
| NUTSLevel | 0 = country, 1 = NUTS 1, 2 = NUTS 2 |
| ParentRegion | NUTS 2 → its NUTS 1 region; NUTS 1 → its country; empty for countries |
| Country | Country the region belongs to |
| Year | 2018–2025 |
| Month | January … December, or `Total` (Eurostat's annual total) |
| ResidenceCountry | `Domestic`, `Foreign` or `Total` (Eurostat's total over all guests) |
| Value | Nights spent |
| PrcCities, PrcRural, PrcCoastal, PrcInlandMountain, PrcIsland | Geography of the region: % of its residents living in cities, rural areas, coastal municipalities, inland mountain and island NUTS 3 regions (step 4b) |

Steps: 1 sheet catalogue · 2 parse · 3 hierarchy · 4 NUTS codes and parent regions · 4b geography · 5 long format and cleaning · 6 checks and corrections · 7 save · 8 test against the source files.

In [1]:
import re
import time
import unicodedata
import warnings

import numpy as np
import openpyxl
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_rows", 200)

SRC = "data/raw/tour_ce_omn12__custom_23247212_spreadsheet.xlsx"   # nights spent (tour_ce_omn12)
NF = "data/raw/NUTS2021-NUTS2024.xlsx"                             # NUTS 2024 codes and labels
OUT = "data/transformed/tour_ce_omn12_long.csv"

sheets = pd.read_excel(SRC, sheet_name=None, header=None)
print(len(sheets), "sheets:", list(sheets))

29 sheets: ['Summary', 'Structure', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7', 'Sheet 8', 'Sheet 9', 'Sheet 10', 'Sheet 11', 'Sheet 12', 'Sheet 13', 'Sheet 14', 'Sheet 15', 'Sheet 16', 'Sheet 17', 'Sheet 18', 'Sheet 19', 'Sheet 20', 'Sheet 21', 'Sheet 22', 'Sheet 23', 'Sheet 24', 'Sheet 25', 'Sheet 26', 'Sheet 27']


## 1. Sheet catalogue from the *Summary* sheet
Maps every data sheet to its residence category and year, so nothing is hard-coded.

In [2]:
summary = sheets["Summary"]
hdr = summary.index[summary[1].eq("Contents")][0]
catalog = summary.iloc[hdr + 1:, 1:7].dropna(subset=[1])
catalog.columns = ["Sheet", "Frequency", "Indicator", "Residence", "Unit", "Year"]
catalog["Year"] = catalog["Year"].astype(int)
RESIDENCE = {"Domestic country": "Domestic", "Foreign country": "Foreign", "Total": "Total"}
catalog["ResidenceCountry"] = catalog["Residence"].map(RESIDENCE)
catalog.reset_index(drop=True)

,Sheet,Frequency,Indicator,Residence,Unit,Year,ResidenceCountry
0,Sheet 1,Annual,Nights spent,Domestic country,Number,2018,Domestic
1,Sheet 2,Annual,Nights spent,Domestic country,Number,2019,Domestic
2,Sheet 3,Annual,Nights spent,Domestic country,Number,2020,Domestic
3,Sheet 4,Annual,Nights spent,Domestic country,Number,2021,Domestic
4,Sheet 5,Annual,Nights spent,Domestic country,Number,2022,Domestic
5,Sheet 6,Annual,Nights spent,Domestic country,Number,2023,Domestic
6,Sheet 7,Annual,Nights spent,Domestic country,Number,2024,Domestic
7,Sheet 8,Annual,Nights spent,Domestic country,Number,2025,Domestic
8,Sheet 9,Annual,Nights spent,Domestic country,Number,2026,Domestic
9,Sheet 10,Annual,Nights spent,Foreign country,Number,2018,Foreign


## 2. Parse one data sheet

In [3]:
def parse_sheet(df):
    month_row = df.index[df[0].eq("MONTH (Labels)")][0]
    geo_row = df.index[df[0].eq("GEO (Labels)")][0]
    end_row = df.index[df[0].eq("Special value")][0]
    block = df.loc[geo_row + 1:end_row - 1].dropna(subset=[0])

    month_cols = {c: df.at[month_row, c] for c in df.columns[1:]
                  if isinstance(df.at[month_row, c], str)}      # value columns only (flag columns are unlabeled)
    values = block[list(month_cols)].rename(columns=month_cols)
    values = values.apply(pd.to_numeric, errors="coerce")       # ':' -> NaN
    values.insert(0, "Label", block[0].str.strip().values)
    values.insert(0, "RowPos", range(len(values)))              # position keeps duplicate names apart
    return values.reset_index(drop=True)

parsed = {s: parse_sheet(sheets[s]) for s in catalog["Sheet"]}

labels = parsed["Sheet 1"]["Label"]
assert all(p["Label"].equals(labels) for p in parsed.values())   # all sheets list the same regions in the same order
MONTHS = [c for c in parsed["Sheet 1"].columns if c not in ("RowPos", "Label")]   # "Total", "January", ...
MONTH_NAMES = MONTHS[1:]
print(len(labels), "geo rows;", MONTHS)

394 geo rows; ['Total', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']


## 3. Country and NUTS level for every row

Eurostat lists geography hierarchically: `EU27 → country → NUTS 1 → its NUTS 2 regions → next NUTS 1 → …`. Country rows are found by walking through the country list in order. Inside a country, the first row is a NUTS 1 region and the following rows are its NUTS 2 children until their running sum covers the NUTS 1 value in every sheet × month cell (tolerance 2 nights; "covers" rather than "equals" because the Netherlands and Portugal also list superseded NUTS 2021 regions).

In [4]:
# Countries in Eurostat's order, with their NUTS country codes (Greece = EL).
CC = {"Belgium": "BE", "Bulgaria": "BG", "Czechia": "CZ", "Denmark": "DK", "Germany": "DE", "Estonia": "EE",
      "Ireland": "IE", "Greece": "EL", "Spain": "ES", "France": "FR", "Croatia": "HR", "Italy": "IT", "Cyprus": "CY",
      "Latvia": "LV", "Lithuania": "LT", "Luxembourg": "LU", "Hungary": "HU", "Malta": "MT", "Netherlands": "NL",
      "Austria": "AT", "Poland": "PL", "Portugal": "PT", "Romania": "RO", "Slovenia": "SI", "Slovakia": "SK",
      "Finland": "FI", "Sweden": "SE", "Iceland": "IS", "Liechtenstein": "LI", "Norway": "NO", "Switzerland": "CH"}
COUNTRIES = list(CC)

mat = np.nan_to_num(np.column_stack([parsed[s][MONTHS].to_numpy(float) for s in catalog["Sheet"]]))
TOL = 2
geo = pd.DataFrame({"RowPos": range(len(labels)), "Region": labels, "Country": None, "NUTSLevel": pd.NA})

blocks, nxt = [], 0
for i, lab in enumerate(labels):
    if nxt < len(COUNTRIES) and lab == COUNTRIES[nxt]:
        blocks.append((COUNTRIES[nxt], i, []))
        nxt += 1
    elif blocks:
        blocks[-1][2].append(i)
assert nxt == len(COUNTRIES), "not all countries found"

for country, crow, rows in blocks:
    geo.loc[crow, ["Country", "NUTSLevel"]] = [country, 0]
    j = 0
    while j < len(rows):
        parent = rows[j]
        geo.loc[parent, ["Country", "NUTSLevel"]] = [country, 1]
        j += 1
        acc, n_children = np.zeros(mat.shape[1]), 0
        while j < len(rows):
            if n_children and (acc >= mat[parent] - TOL).all():
                break
            acc += mat[rows[j]]
            geo.loc[rows[j], ["Country", "NUTSLevel"]] = [country, 2]
            j += 1
            n_children += 1
        assert (acc >= mat[parent] - TOL).all(), f"children do not add up for {labels[parent]}"

print(geo["NUTSLevel"].value_counts(dropna=False).sort_index())   # NaN = EU27 aggregate

NUTSLevel
0        31
1        97
2       265
<NA>      1
Name: count, dtype: int64


## 4. NUTS codes and parent regions

The tourism spreadsheet has region names only, and names repeat across levels (*Berlin*, *Malta*, *Luxembourg*). Each region row is therefore given its **NUTS 2024 code** here, before anything else, so every later step can use one unique key.

* Codes come from `data/raw/NUTS2021-NUTS2024.xlsx` (sheets *NUTS2024*, and *Statistical Regions* for Iceland, Liechtenstein, Norway, Switzerland), matched on country + level + name. Bulgarian, Greek and Cypriot names are in their own scripts there, so the file's *Cyrillic & Greek to Latin* sheet is used; names are compared case-insensitively with dashes and spaces normalised, and as a fallback on the part before a "/" (*Trøndelag/Trööndelage*).
* Rows left out of the dataset are not coded: the EU27 aggregate and the superseded **NUTS 2021** regions of the Netherlands and Portugal (labelled "(NUTS 2021)"); only the NUTS 2024 version is kept.
* **ParentRegion**: a NUTS 2 region's parent is the NUTS 1 region whose code is the first three characters of its code; a NUTS 1 region's parent is its country.

In [5]:
nfx = pd.ExcelFile(NF)                                    # opened once, read sheet by sheet
nuts = pd.read_excel(nfx, sheet_name="NUTS2024")
sr = pd.read_excel(nfx, sheet_name="Statistical Regions")
sr.columns = nuts.columns
codes = pd.concat([nuts, sr], ignore_index=True)
codes.columns = ["cc", "NUTSCode", "label", "NUTSLevel", "order", "num"]
codes["NUTSCode"] = codes["NUTSCode"].astype(str).str.strip()
latin = pd.read_excel(nfx, sheet_name="Cyrillic & Greek to Latin")
latin.columns = ["NUTSCode", "label", "latin"]
latin["NUTSCode"] = latin["NUTSCode"].astype(str).str.strip()
codes = codes.merge(latin[["NUTSCode", "latin"]], on="NUTSCode", how="left")
codes["label_latin"] = codes["latin"].fillna(codes["label"])

def norm(s):
    s = unicodedata.normalize("NFC", str(s)).strip().casefold()
    s = re.sub(r"[-–—]", " ", s)
    return re.sub(r"\s+", " ", s)

codes["key"] = codes["label_latin"].map(norm)
codes["key_first"] = codes["label_latin"].map(lambda s: norm(str(s).split("/")[0]))

keep = geo["NUTSLevel"].notna() & ~geo["Region"].str.endswith("(NUTS 2021)")
print("left out:", geo.loc[~keep, "Region"].tolist())
g = geo[keep].copy()
g["NUTSLevel"] = g["NUTSLevel"].astype(int)
g["cc"] = g["Country"].map(CC)
g["key"] = g["Region"].map(norm)
lvl12 = codes[codes["NUTSLevel"].isin([1, 2])]
g = g.merge(lvl12[["cc", "NUTSLevel", "key", "NUTSCode"]], on=["cc", "NUTSLevel", "key"], how="left")
miss = g["NUTSCode"].isna() & (g["NUTSLevel"] > 0)
fb = g.loc[miss, ["cc", "NUTSLevel", "key"]].merge(
    lvl12[["cc", "NUTSLevel", "key_first", "NUTSCode"]].rename(columns={"key_first": "key"}), on=["cc", "NUTSLevel", "key"], how="left")
g.loc[miss, "NUTSCode"] = fb["NUTSCode"].values
g.loc[g["NUTSLevel"] == 0, "NUTSCode"] = g["cc"]
print("matched via fallback:", g.loc[miss, ["Region", "NUTSCode"]].values.tolist())

# duplicate rows of the same region (e.g. 'Berlin' at NUTS 1 and NUTS 2 is two regions, DE3 and DE30)
assert g["NUTSCode"].notna().all(), g[g["NUTSCode"].isna()]
assert g["NUTSCode"].is_unique
assert (g["NUTSCode"].str.len() == g["NUTSLevel"].map({0: 2, 1: 3, 2: 4})).all()
assert g.loc[g["NUTSLevel"] == 2, "NUTSCode"].str[:3].isin(set(g.loc[g["NUTSLevel"] == 1, "NUTSCode"])).all()

name_of = g.set_index("NUTSCode")["Region"]
g["ParentRegion"] = np.where(g["NUTSLevel"] == 2, g["NUTSCode"].str[:3].map(name_of),
                             np.where(g["NUTSLevel"] == 1, g["Country"], None))
print(f"{len(g)} regions coded:", g["NUTSLevel"].value_counts().sort_index().to_dict())
g[["Region", "NUTSCode", "NUTSLevel", "ParentRegion", "Country"]].query("Country in ['Germany', 'Greece']").head(10)

left out: ['European Union - 27 countries (from 2020)', 'Utrecht (NUTS 2021)', 'Zuid-Holland (NUTS 2021)', 'Centro (PT) (NUTS 2021)', 'Área Metropolitana de Lisboa (NUTS 2021)', 'Alentejo (NUTS 2021)']
matched via fallback: [['Trøndelag', 'NO06']]
388 regions coded: {0: 31, 1: 97, 2: 260}


,Region,NUTSCode,NUTSLevel,ParentRegion,Country
41,Germany,DE,0,NaN,Germany
42,Baden-Württemberg,DE1,1,Germany,Germany
43,Stuttgart,DE11,2,Baden-Württemberg,Germany
44,Karlsruhe,DE12,2,Baden-Württemberg,Germany
45,Freiburg,DE13,2,Baden-Württemberg,Germany
46,Tübingen,DE14,2,Baden-Württemberg,Germany
47,Bayern,DE2,1,Germany,Germany
48,Oberbayern,DE21,2,Bayern,Germany
49,Niederbayern,DE22,2,Bayern,Germany
50,Oberpfalz,DE23,2,Bayern,Germany


## 4b. Geography of the regions (municipality and NUTS 3 data)

Shares of each region's **residents** (2024 population) living in particular kinds of places, from two Eurostat files:

* `data/raw/EU-27-LAU-2024-NUTS-2024.xlsx` — every municipality (LAU) with its NUTS 3 code, population, degree of urbanisation (DEGURBA: 1 cities, 2 towns and suburbs, 3 rural areas) and coastal flag;
* `data/raw/NUTS2021-NUTS2024.xlsx`, sheet *NUTS-3 Typologies* — each NUTS 3 region classed as mountain or not, coastal or not, island or not.

| Column | Meaning |
|---|---|
| PrcCities / PrcRural | % living in cities / rural areas (towns and suburbs = the rest) |
| PrcCoastal | % living in coastal municipalities |
| PrcInlandMountain | % living in NUTS 3 regions that are mountain **and** not coastal (Alps, Pyrenees, Carpathians …; Eurostat's broader mountain class also covers most Mediterranean coasts and islands) |
| PrcIsland | % living in island NUTS 3 regions |

Municipal population is summed up the hierarchy (NUTS 3 code → first 4 characters = NUTS 2, first 3 = NUTS 1, first 2 = country). The same values apply to every year. No values for Iceland (not in the municipality file); no mountain/island values for Liechtenstein and Greece's Extra-Regio area (not in the typology sheet). Municipalities without population or with DEGURBA 9 (northern Cyprus) are left out.

In [6]:
LF = "data/raw/EU-27-LAU-2024-NUTS-2024.xlsx"
xl = pd.ExcelFile(LF)
lau_sheets = [s for s in xl.sheet_names if s in CC.values()]
lau = pd.concat([pd.read_excel(xl, sheet_name=s, usecols=["NUTS3", "POPULATION", "DEGURBA", "COASTAL AREA"])
                 for s in lau_sheets], ignore_index=True)
lau["POPULATION"] = pd.to_numeric(lau["POPULATION"], errors="coerce")
n_all = len(lau)
lau = lau[lau["DEGURBA"].isin([1, 2, 3]) & lau["POPULATION"].notna()].copy()
lau["NUTS3"] = lau["NUTS3"].astype(str).str.strip()
print(f"{len(lau_sheets)} countries, {n_all:,} municipalities, {len(lau):,} used, {lau['POPULATION'].sum() / 1e6:.1f} m residents")

typ = pd.read_excel(nfx, sheet_name="NUTS-3 Typologies")
typ["NUTS3"] = typ["NUTS-3 Code"].astype(str).str.strip()
known = typ["Mountanious regions"].notna()
typ["inland_mountain"] = np.where(known, (typ["Mountanious regions"].astype(str).str.startswith("mountain")
                                          & (typ["Coastal regions"] == "non-coastal")).astype(float), np.nan)
typ["island"] = np.where(typ["Island regions"].notna(), (typ["Island regions"] == "Island").astype(float), np.nan)
pop3 = lau.groupby("NUTS3")["POPULATION"].sum().rename("pop").reset_index().merge(
    typ[["NUTS3", "inland_mountain", "island"]], on="NUTS3", how="left")

parts = []
for n_chars in (2, 3, 4):                                   # country, NUTS 1, NUTS 2
    l_ = lau.assign(code=lau["NUTS3"].str[:n_chars])
    p = l_.pivot_table(index="code", columns="DEGURBA", values="POPULATION", aggfunc="sum").reindex(columns=[1, 2, 3]).fillna(0)
    c_ = l_.assign(cp=l_["POPULATION"] * l_["COASTAL AREA"].fillna(0)).groupby("code")[["cp", "POPULATION"]].sum()
    out = pd.DataFrame({"PrcCities": 100 * p[1] / p.sum(axis=1), "PrcRural": 100 * p[3] / p.sum(axis=1),
                        "PrcCoastal": 100 * c_["cp"] / c_["POPULATION"]})
    for flag, col in (("inland_mountain", "PrcInlandMountain"), ("island", "PrcIsland")):
        q = pop3.dropna(subset=[flag]).assign(code=lambda d: d["NUTS3"].str[:n_chars])
        q = q.assign(f=q["pop"] * q[flag]).groupby("code")[["f", "pop"]].sum()
        out[col] = 100 * q["f"] / q["pop"]
    parts.append(out)
GEO_COLS = ["PrcCities", "PrcRural", "PrcCoastal", "PrcInlandMountain", "PrcIsland"]
geo_shares = pd.concat(parts).round(2)[GEO_COLS]
g = g.merge(geo_shares, left_on="NUTSCode", right_index=True, how="left")
print("regions without any geography values:", g.loc[g["PrcCities"].isna(), "Region"].tolist())
print("regions without mountain/island values:", g.loc[g["PrcIsland"].isna(), "Region"].tolist())
g[g["NUTSLevel"] == 2][["Region", "NUTSCode"] + GEO_COLS].sort_values("PrcInlandMountain", ascending=False).head(6)

30 countries, 97,597 municipalities, 97,408 used, 467.4 m residents
regions without any geography values: ['Iceland', 'Ísland', 'Ísland']
regions without mountain/island values: ['Extra-Regio NUTS 1', 'Extra-Regio NUTS 2', 'Iceland', 'Ísland', 'Ísland', 'Liechtenstein', 'Liechtenstein', 'Liechtenstein']


,Region,NUTSCode,PrcCities,PrcRural,PrcCoastal,PrcInlandMountain,PrcIsland
387,Ticino,CH07,29.09,13.03,0.0,100.0,0.0
22,Yugozapaden,BG41,67.79,9.59,0.0,100.0,0.0
114,Dytiki Makedonia,EL53,0.00,48.09,0.0,100.0,0.0
131,Comunidad Foral de Navarra,ES22,50.64,32.22,0.0,100.0,0.0
132,La Rioja,ES23,46.60,23.34,0.0,100.0,0.0
133,Aragón,ES24,55.03,23.70,0.0,100.0,0.0


## 5. Long format and cleaning

* One row per region × year × month (incl. the annual `Total`) × residence category.
* **2026 is removed**: the extract only has January–March 2026 and no annual total.
* **`:` (not available) is replaced with 0** wherever the region has published values for that year and residence: in all such series the published months add up exactly to the published annual total, so these `:` behave as zero nights. Series where *every* cell of the year is `:` — the region was not published that year (the Dutch and Portuguese NUTS 2024 regions before 2024, Greece's Extra-Regio area 2018–2024, Iceland's region *Ísland* in 2025) — are left out, because 0 would claim the region had no guests.

In [7]:
frames = []
for _, meta in catalog.iterrows():
    long = parsed[meta["Sheet"]].melt(id_vars=["RowPos"], value_vars=MONTHS, var_name="Month", value_name="Value")
    long["Year"] = meta["Year"]
    long["ResidenceCountry"] = meta["ResidenceCountry"]
    frames.append(long)
df = pd.concat(frames, ignore_index=True).merge(g[["RowPos", "Region", "NUTSCode", "NUTSLevel", "ParentRegion", "Country"] + GEO_COLS],
                                                on="RowPos", how="inner")      # drops EU27 and NUTS 2021 rows
print(f"{len(df):,} cells")

is2026 = df["Year"] == 2026
print(f"dropping 2026: {is2026.sum():,} cells")
df = df[~is2026]

KEY_SERIES = ["NUTSCode", "Year", "ResidenceCountry"]
published = df["Value"].notna().groupby([df[k] for k in KEY_SERIES]).transform("any")
missing = df["Value"].isna()
print(f"':' replaced with 0: {(missing & published).sum():,} cells in {df[missing & published].groupby(KEY_SERIES).ngroups} series")
unpub = df[~published]
print(f"series with no published value, left out: {unpub.groupby(KEY_SERIES).ngroups} ({len(unpub):,} cells)")
display(unpub.groupby(["Region", "NUTSCode"]).agg(years=("Year", lambda y: f"{min(y)}-{max(y)}")).reset_index())
df = df[published].copy()
df["Value"] = df["Value"].fillna(0).round().astype("int64")

df["Month"] = pd.Categorical(df["Month"], categories=MONTH_NAMES + ["Total"], ordered=True)
df["Country"] = pd.Categorical(df["Country"], categories=COUNTRIES, ordered=True)
df = df.sort_values(["Country", "RowPos", "Year", "ResidenceCountry", "Month"]).reset_index(drop=True)
KEY = ["NUTSCode", "Year", "Month", "ResidenceCountry"]
assert not df.duplicated(KEY).any()
print(f"{len(df):,} rows; years {df['Year'].min()}-{df['Year'].max()}")

136,188 cells
dropping 2026: 15,132 cells
':' replaced with 0: 51 cells in 31 series
series with no published value, left out: 174 (2,262 cells)


,Region,NUTSCode,years
0,Alentejo,PT1C,2018-2023
1,Centro (PT),PT19,2018-2023
2,Extra-Regio NUTS 1,ELZ,2018-2024
3,Extra-Regio NUTS 2,ELZZ,2018-2024
4,Grande Lisboa,PT1A,2018-2023
5,Oeste e Vale do Tejo,PT1D,2018-2023
6,Península de Setúbal,PT1B,2018-2023
7,Utrecht,NL35,2018-2023
8,Zuid-Holland,NL36,2018-2023
9,Ísland,IS0,2025-2025


118,794 rows; years 2018-2025


## 6. Checks and corrections

**Annual total vs. months:** the published annual `Total` should equal the sum of the 12 months.

**Country vs. its regions:** a country's value should be at least the sum of its regions. *Country > regions* is possible (nights not allocated to a region, and the missing Dutch and Portuguese NUTS 2 regions before 2024); *regions > country* should not happen.

**Correction rule:** in a **one-region country** (one NUTS 1 and one NUTS 2 region: Estonia, Cyprus, Latvia, Luxembourg, Malta, Iceland, Liechtenstein) the regions *are* the country, so a regional value **above** the country value is replaced by the country value. Where a monthly value was corrected, the region's annual `Total` is recomputed as the sum of its 12 months.

**Known source error, kept as published:** *Latvija* 2024 (NUTS 1 and 2), Foreign and Total — annual total 268 and October–December near zero, while the Latvia country row shows 1.33 million foreign nights. The regional values are *below* the country, so the rule does not apply; use the country row for Latvia 2024.

In [8]:
def totals_vs_months(data):
    m = data[data["Month"] != "Total"].groupby(KEY_SERIES, observed=True)["Value"].agg(["sum", "count"])
    t = data[data["Month"] == "Total"].set_index(KEY_SERIES)["Value"].rename("total")
    c = m[m["count"] == 12].join(t, how="inner")
    c["diff"] = c["total"] - c["sum"]
    return c[c["diff"] != 0]

KEY_COUNTRY = ["Country", "Year", "Month", "ResidenceCountry"]

def country_vs_regions(data):
    country = data[data["NUTSLevel"] == 0].set_index(KEY_COUNTRY)["Value"].rename("country")
    out = {}
    for lvl in (1, 2):
        reg = data[data["NUTSLevel"] == lvl].groupby(KEY_COUNTRY, observed=True)["Value"].sum().rename("regions")
        j = pd.concat([country, reg], axis=1).dropna()
        j["diff"] = j["country"] - j["regions"]
        out[lvl] = j
        print(f"NUTS {lvl}: {len(j):,} cells | equal {(j['diff'] == 0).sum():,} | country > regions {(j['diff'] > 0).sum():,} "
              f"| regions > country {(j['diff'] < 0).sum():,}")
    return out

print("annual total != sum of months:")
display(totals_vs_months(df).reset_index().merge(g[["NUTSCode", "Region"]], on="NUTSCode"))
gaps = country_vs_regions(df)
display(gaps[1][gaps[1]["diff"] < 0])

annual total != sum of months:


,NUTSCode,Year,ResidenceCountry,sum,count,total,diff,Region
0,LV0,2024,Foreign,1103522,12,268,-1103254,Latvija
1,LV0,2024,Total,1446182,12,268,-1445914,Latvija
2,LV00,2024,Foreign,1103522,12,268,-1103254,Latvija
3,LV00,2024,Total,1446182,12,268,-1445914,Latvija


NUTS 1: 9,633 cells | equal 9,191 | country > regions 439 | regions > country 3
NUTS 2: 9,633 cells | equal 8,723 | country > regions 907 | regions > country 3


country   regions  diff
Country Year Month    ResidenceCountry                         
Iceland 2024 November Domestic             4887    4893.0  -6.0
             December Domestic             3883    3889.0  -6.0
             October  Foreign            226672  226685.0 -13.0

In [9]:
n_reg = df[df["NUTSLevel"] > 0].groupby(["Country", "NUTSLevel"], observed=True)["NUTSCode"].nunique().unstack()
ONE_REGION = n_reg[(n_reg[1] == 1) & (n_reg[2] == 1)].index.tolist()
print("one-region countries:", ONE_REGION)

df = df.join(df[df["NUTSLevel"] == 0].set_index(KEY_COUNTRY)["Value"].rename("Value_country"), on=KEY_COUNTRY)
fix = df["Country"].isin(ONE_REGION) & (df["NUTSLevel"] > 0) & (df["Value"] > df["Value_country"])
changes = df.loc[fix, ["Region", "NUTSLevel", "Country"] + KEY + ["Value", "Value_country"]].copy()
print(f"{fix.sum()} regional cells corrected (regions > country):")
display(changes.reset_index(drop=True))
df.loc[fix, "Value"] = df.loc[fix, "Value_country"]
df = df.drop(columns="Value_country")

touched = changes.loc[changes["Month"] != "Total", KEY_SERIES].drop_duplicates()
msum = (df[df["Month"] != "Total"].merge(touched, on=KEY_SERIES)
        .groupby(KEY_SERIES, observed=True)["Value"].agg(month_sum="sum", n="count").reset_index())
tot = df[df["Month"] == "Total"].reset_index().merge(msum[msum["n"] == 12], on=KEY_SERIES)
tot = tot[tot["Value"] != tot["month_sum"]]
df.loc[tot["index"], "Value"] = tot["month_sum"].values
print(f"{len(tot)} annual totals recomputed:")
display(tot[["Region", "NUTSCode", "Year", "ResidenceCountry", "Value", "month_sum"]]
        .rename(columns={"Value": "Total_before", "month_sum": "Total_after"}))
df["Value"] = df["Value"].astype("int64")

print("after correction:")
country_vs_regions(df)
print("annual total != sum of months (Latvija 2024 expected):")
display(totals_vs_months(df).reset_index().merge(g[["NUTSCode", "Region"]], on="NUTSCode"))

one-region countries: ['Estonia', 'Cyprus', 'Latvia', 'Luxembourg', 'Malta', 'Iceland', 'Liechtenstein']
6 regional cells corrected (regions > country):


,Region,NUTSLevel,Country,NUTSCode,Year,Month,ResidenceCountry,Value,Value_country
0,Ísland,1,Iceland,IS0,2024,November,Domestic,4893,4887
1,Ísland,1,Iceland,IS0,2024,December,Domestic,3889,3883
2,Ísland,1,Iceland,IS0,2024,October,Foreign,226685,226672
3,Ísland,2,Iceland,IS00,2024,November,Domestic,4893,4887
4,Ísland,2,Iceland,IS00,2024,December,Domestic,3889,3883
5,Ísland,2,Iceland,IS00,2024,October,Foreign,226685,226672


4 annual totals recomputed:


,Region,NUTSCode,Year,ResidenceCountry,Total_before,Total_after
0,Ísland,IS0,2024,Domestic,88889,88877
1,Ísland,IS0,2024,Foreign,2437306,2437293
2,Ísland,IS00,2024,Domestic,88889,88877
3,Ísland,IS00,2024,Foreign,2437306,2437293


after correction:
NUTS 1: 9,633 cells | equal 9,194 | country > regions 439 | regions > country 0
NUTS 2: 9,633 cells | equal 8,726 | country > regions 907 | regions > country 0
annual total != sum of months (Latvija 2024 expected):


,NUTSCode,Year,ResidenceCountry,sum,count,total,diff,Region
0,LV0,2024,Foreign,1103522,12,268,-1103254,Latvija
1,LV0,2024,Total,1446182,12,268,-1445914,Latvija
2,LV00,2024,Foreign,1103522,12,268,-1103254,Latvija
3,LV00,2024,Total,1446182,12,268,-1445914,Latvija


## 7. Save

In [10]:
FINAL_COLS = ["Region", "NUTSCode", "NUTSLevel", "ParentRegion", "Country", "Year", "Month", "ResidenceCountry", "Value"] + GEO_COLS
df = df[FINAL_COLS]
df.to_csv(OUT, index=False, encoding="utf-8-sig")   # BOM so Excel shows accented names correctly
print(f"saved {OUT}: {len(df):,} rows x {df.shape[1]} columns")
print(df.groupby(["NUTSLevel"])["NUTSCode"].nunique().rename("regions").to_dict(), "| countries:", df["Country"].nunique())

saved data/transformed/tour_ce_omn12_long.csv: 118,794 rows x 14 columns
{0: 31, 1: 97, 2: 260} | countries: 31


## 8. Test: random comparison with the source files

The saved CSV is **read back from disk** and random samples are compared with **independent reads** of the source files: the tourism workbook is opened cell by cell with `openpyxl` (not the pandas parser of step 2), regions are located by name inside their country block, codes and parent regions are looked up in the NUTS file again, and the geography shares are recomputed region by region from freshly read municipality and typology sheets. Expected differences: `:` is 0 inside a published series and absent where the whole series is unpublished; the corrected cells of step 6 carry their corrected value (checked separately in test D). The seed is printed, so a failing run can be repeated.

In [11]:
SEED = int(time.time()) % 100_000            # replace with a fixed number to repeat a run
rng = np.random.default_rng(SEED)
print("random seed:", SEED)

final = pd.read_csv(OUT, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
csv_value = final.set_index(["Region", "NUTSLevel", "Country", "Year", "Month", "ResidenceCountry"])["Value"]
assert csv_value.index.is_unique
RESIDENCES = ["Domestic", "Foreign", "Total"]

def key_of(r):
    return (r["Region"], int(r["NUTSLevel"]), str(r["Country"]), int(r["Year"]), str(r["Month"]), str(r["ResidenceCountry"]))

corrected = {key_of(r): int(r["Value_country"]) for _, r in changes.iterrows()}
corrected.update({key_of(r): int(r["month_sum"]) for _, r in tot.iterrows()})
print(len(corrected), "documented corrections")

results = []
def report(name, n, bad):
    results.append({"test": name, "checked": n, "mismatches": len(bad)})
    print(f"{name}: {n} checked, {len(bad)} mismatches")
    if bad:
        display(pd.DataFrame(bad).head(20))

wb = openpyxl.load_workbook(SRC, data_only=True)
sheet_of = {}
for row in wb["Summary"].iter_rows(values_only=True):
    if len(row) > 6 and isinstance(row[1], str) and row[1].startswith("Sheet ") and row[4] in RESIDENCE:
        sheet_of[(RESIDENCE[row[4]], int(row[6]))] = row[1]
assert len(sheet_of) == 27

_layout = {}
def layout(sheet):
    if sheet not in _layout:
        rows = list(wb[sheet].iter_rows(values_only=True))
        labs = [str(r[0]).strip() if r[0] is not None else None for r in rows]
        mrow = labs.index("MONTH (Labels)")
        months = {str(v).strip(): j for j, v in enumerate(rows[mrow]) if j > 0 and v not in (None, "")}
        _layout[sheet] = (rows, labs, months)
    return _layout[sheet]

def country_block(labs, country):
    start = labs.index(country)                          # first occurrence = the country row
    i = COUNTRIES.index(country) + 1
    end = labs.index(COUNTRIES[i], start + 1) if i < len(COUNTRIES) else labs.index("Special value")
    return start, end

def find_row(labs, region, level, country):
    start, end = country_block(labs, country)
    if level == 0:
        return start
    hits = [i for i in range(start + 1, end) if labs[i] == region]
    if len(hits) == 1:
        return hits[0]
    if len(hits) == 2:                                   # same name at NUTS 1 and NUTS 2: NUTS 1 comes first
        return hits[0] if level == 1 else hits[1]
    raise ValueError((region, level, country, hits))

def raw_value(region, level, country, year, month, res):
    rows, labs, months = layout(sheet_of[(res, year)])
    v = rows[find_row(labs, region, level, country)][months[month]]
    try:
        return float(v)
    except (TypeError, ValueError):
        return None                                      # ':'

def series_unpublished(region, level, country, year, res):
    return all(raw_value(region, level, country, year, m, res) is None for m in MONTHS)

def source_value(k):
    if k in corrected:
        return corrected[k]
    v = raw_value(*k)
    if v is None and not series_unpublished(k[0], k[1], k[2], k[3], k[5]):
        return 0.0
    return v

random seed: 97203
10 documented corrections


In [12]:
# A. Random CSV rows -> value in the tourism workbook
bad = []
samp = final.sample(500, random_state=SEED)
for _, r in samp.iterrows():
    k = key_of(r)
    exp = source_value(k)
    if exp is None or round(exp) != r["Value"]:
        bad.append({"key": k, "csv": r["Value"], "source": exp})
report("A. CSV value vs tourism workbook", len(samp), bad)

# B. Random workbook cells -> present in the CSV with the same value (or absent if the series is unpublished)
levels_of = final[final["NUTSLevel"] > 0].groupby(["Region", "Country"])["NUTSLevel"].unique()
years = sorted({y for (_, y) in sheet_of if y != 2026})
bad, n, tries = [], 0, 0
while n < 500 and tries < 10_000:
    tries += 1
    res, year = str(rng.choice(RESIDENCES)), int(rng.choice(years))
    rows, labs, months = layout(sheet_of[(res, year)])
    country = str(rng.choice(COUNTRIES))
    start, end = country_block(labs, country)
    i = int(rng.integers(start, end))
    region, month = labs[i], str(rng.choice(list(months)))
    if region is None or region.endswith("(NUTS 2021)"):
        continue
    try:
        v = float(rows[i][months[month]])
    except (TypeError, ValueError):
        v = None
    if i == start:
        level = 0
    else:
        hits = [h for h in range(start + 1, end) if labs[h] == region]
        level = (1 if i == hits[0] else 2) if len(hits) == 2 else int(levels_of[(region, country)][0])
    k = (region, level, country, year, month, res)
    if v is None:
        exp = None if series_unpublished(region, level, country, year, res) else 0
    else:
        exp = corrected.get(k, round(v))
    got = csv_value.get(k)
    n += 1
    if got != exp:
        bad.append({"key": k, "source": exp, "csv": got})
report("B. Workbook cell vs CSV value", n, bad)

A. CSV value vs tourism workbook: 500 checked, 0 mismatches
B. Workbook cell vs CSV value: 500 checked, 0 mismatches


In [13]:
# C. NUTS codes and parent regions vs the NUTS file
lab_rows = []
for sh in ["NUTS2024", "Statistical Regions"]:
    t_ = pd.read_excel(NF, sheet_name=sh)
    lab_rows.append(pd.DataFrame({"code": t_.iloc[:, 1].astype(str).str.strip(), "label": t_.iloc[:, 2].astype(str)}))
label_raw = pd.concat(lab_rows).drop_duplicates("code").set_index("code")["label"]
tr_ = pd.read_excel(NF, sheet_name="Cyrillic & Greek to Latin")
translit = dict(zip(tr_.iloc[:, 0].astype(str).str.strip(), tr_.iloc[:, 2]))
label_of = lambda code: translit.get(code, label_raw.get(code))
same_name = lambda label, name: label is not None and (norm(label) == norm(name) or norm(str(label).split("/")[0]) == norm(name))

regs_s = final[final["NUTSLevel"] > 0][["Region", "NUTSCode", "NUTSLevel", "ParentRegion", "Country"]].drop_duplicates()
regs_s = regs_s.sample(100, random_state=SEED)
bad = []
for _, r in regs_s.iterrows():
    code, lvl_ = r["NUTSCode"], int(r["NUTSLevel"])
    ok = len(code) == lvl_ + 2 and code[:2] == CC[r["Country"]] and same_name(label_of(code), r["Region"])
    ok_parent = (r["ParentRegion"] == r["Country"]) if lvl_ == 1 else same_name(label_of(code[:3]), r["ParentRegion"])
    if not (ok and ok_parent):
        bad.append({"Region": r["Region"], "NUTSCode": code, "label": label_of(code), "ParentRegion": r["ParentRegion"]})
report("C. NUTSCode and ParentRegion vs NUTS file", len(regs_s), bad)

C. NUTSCode and ParentRegion vs NUTS file: 100 checked, 0 mismatches


In [14]:
# D. Corrections: in the file, and corrected months equal the published country value
bad = []
for k, v in corrected.items():
    if csv_value.get(k) != v:
        bad.append({"key": k, "expected": v, "csv": csv_value.get(k)})
for _, r in changes.iterrows():
    k = key_of(r)
    country_raw = raw_value(k[2], 0, k[2], k[3], k[4], k[5])
    if country_raw is None or round(country_raw) != corrected[k]:
        bad.append({"key": k, "country (published)": country_raw, "corrected": corrected[k]})
report("D. Documented corrections", len(corrected), bad)

# E. Structure of the saved file
checks = {
    "column order": list(final.columns) == FINAL_COLS,
    "no 2026": not (final["Year"] == 2026).any(),
    "no NUTS 2021 regions": not final["Region"].str.endswith("(NUTS 2021)").any(),
    "unique key NUTSCode x Year x Month x Residence": not final.duplicated(["NUTSCode", "Year", "Month", "ResidenceCountry"]).any(),
    "one name, level, country and geography per code": (final.groupby("NUTSCode")[["Region", "NUTSLevel", "Country"] + GEO_COLS]
                                                          .nunique(dropna=False) == 1).all().all(),
    "geography shares within 0-100": final[GEO_COLS].stack().dropna().between(0, 100).all(),   # empty = no data (e.g. Iceland)
    "ParentRegion empty exactly on country rows": (final["ParentRegion"].isna() == (final["NUTSLevel"] == 0)).all(),
    "no empty or negative values": final["Value"].notna().all() and (final["Value"] >= 0).all(),
    "13 rows (12 months + Total) per series": (final.groupby(["NUTSCode", "Year", "ResidenceCountry"]).size() == 13).all(),
}
bad = [{"check": c} for c, ok in checks.items() if not ok]
report("E. File structure", len(checks), bad)

D. Documented corrections: 10 checked, 0 mismatches
E. File structure: 9 checked, 0 mismatches


In [15]:
# F. Geography shares recomputed region by region from freshly read municipality and typology sheets
typ_f = pd.read_excel(NF, sheet_name="NUTS-3 Typologies")
typ_f.index = typ_f.iloc[:, 1].astype(str).str.strip()
_lau_cache = {}
def lau_rows(cc):
    if cc not in _lau_cache:
        t_ = pd.read_excel(xl, sheet_name=cc, usecols=["NUTS3", "POPULATION", "DEGURBA", "COASTAL AREA"])
        t_["POPULATION"] = pd.to_numeric(t_["POPULATION"], errors="coerce")
        t_["NUTS3"] = t_["NUTS3"].astype(str).str.strip()
        _lau_cache[cc] = t_[t_["DEGURBA"].isin([1, 2, 3]) & t_["POPULATION"].notna()]
    return _lau_cache[cc]

pct = lambda num, den: 100 * num / den if den > 0 else np.nan
regs_f = final.drop_duplicates("NUTSCode").sample(40, random_state=SEED)
bad = []
for _, r in regs_f.iterrows():
    code, cc = r["NUTSCode"], r["NUTSCode"][:2]
    if cc not in xl.sheet_names:
        exp = dict.fromkeys(GEO_COLS, np.nan)
    else:
        sub = lau_rows(cc)
        sub = sub[sub["NUTS3"].str.startswith(code)]
        pop, tot_pop = sub["POPULATION"], sub["POPULATION"].sum()
        exp = {"PrcCities": pct(pop[sub["DEGURBA"] == 1].sum(), tot_pop), "PrcRural": pct(pop[sub["DEGURBA"] == 3].sum(), tot_pop),
               "PrcCoastal": pct((pop * sub["COASTAL AREA"].fillna(0)).sum(), tot_pop)}
        p3 = sub.groupby("NUTS3")["POPULATION"].sum()
        t3 = typ_f.reindex(p3.index)
        m_known, i_known = t3["Mountanious regions"].notna(), t3["Island regions"].notna()
        inland = m_known & t3["Mountanious regions"].astype(str).str.startswith("mountain") & (t3["Coastal regions"] == "non-coastal")
        exp["PrcInlandMountain"] = pct(p3[inland].sum(), p3[m_known].sum())
        exp["PrcIsland"] = pct(p3[i_known & (t3["Island regions"] == "Island")].sum(), p3[i_known].sum())
    for col in GEO_COLS:
        e, gv = exp[col], r[col]
        if not ((pd.isna(e) and pd.isna(gv)) or (pd.notna(e) and pd.notna(gv) and abs(e - gv) <= 0.011)):
            bad.append({"Region": r["Region"], "NUTSCode": code, "column": col, "expected": e, "csv": gv})
report("F. Geography shares vs municipality and typology files", len(regs_f), bad)

F. Geography shares vs municipality and typology files: 40 checked, 0 mismatches


In [16]:
summary_tbl = pd.DataFrame(results)
display(summary_tbl)
assert summary_tbl["mismatches"].sum() == 0, "some source checks failed - see the tables above"
print(f"All source checks passed (seed {SEED}).")

,test,checked,mismatches
0,A. CSV value vs tourism workbook,500,0
1,B. Workbook cell vs CSV value,500,0
2,C. NUTSCode and ParentRegion vs NUTS file,100,0
3,D. Documented corrections,10,0
4,E. File structure,9,0
5,F. Geography shares vs municipality and typolo...,40,0


All source checks passed (seed 97203).
